In [1]:
import os, json
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])

print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)

DATA_PATH = FEATURES / "weekly_category.parquet"
print(f"Reading data from: {DATA_PATH}")


FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results
Reading data from: D:\STAT3013.Q12_Group01\features\weekly_category.parquet


In [2]:
import pandas as pd
import numpy as np

wd = pd.read_parquet(FEATURES / "weekly_category.parquet")
wd.columns = wd.columns.str.lower()
forecast_df = wd.sort_values(["group_id", "week_no"]).copy()
forecast_df["time_idx"] = forecast_df["week_no"] - forecast_df["week_no"].min()

# Map cột 'target' dùng cho forecast (ở đây dùng 'sales' làm target)
if "target" not in forecast_df.columns:
    if "sales" in forecast_df.columns:
        forecast_df = forecast_df.rename(columns={"sales": "target"})
    elif "sales_value" in forecast_df.columns:
        forecast_df["target"] = forecast_df["sales_value"]
    elif "qty" in forecast_df.columns:
        forecast_df["target"] = forecast_df["qty"]
    else:
        raise ValueError(
            "Không tìm thấy cột 'sales', 'sales_value' hoặc 'qty' để map sang 'target'."
        )

print("forecast_df shape:", forecast_df.shape)
forecast_df.head()


forecast_df shape: (1083, 16)


,week_no,group_id,qty,target,retail_disc,coupon_disc,coupon_match_disc,avg_price,promo_display,promo_mailer,promo_price_red,total_disc,gross_sales,discount_rate,weekofyear,time_idx
0,1,AUTOMOTIVE,1,4.99,0.0,0.0,0.0,4.990,0.0,0.0,0.0,0.0,4.99,0.0,1,0
148,8,AUTOMOTIVE,1,4.45,0.0,0.0,0.0,4.450,0.0,0.0,0.0,0.0,4.45,0.0,8,7
193,10,AUTOMOTIVE,1,17.99,0.0,0.0,0.0,15.230,0.0,0.0,0.0,0.0,17.99,0.0,10,9
216,11,AUTOMOTIVE,1,0.50,0.0,0.0,0.0,0.500,0.0,0.0,0.0,0.0,0.50,0.0,11,10
264,13,AUTOMOTIVE,2,16.49,0.0,0.0,0.0,8.245,0.0,0.0,0.0,0.0,16.49,0.0,13,12


In [3]:
group_target_sum = (
    forecast_df
    .groupby("group_id")["target"]
    .sum()
    .sort_values(ascending=False)
)

# Lấy top 10 group_id
top_groups = group_target_sum.head(10).index.tolist()

print("Top 10 group_id theo tổng 'target':")
print(group_target_sum.head(10))

# Giữ lại dữ liệu của các group này
forecast_df_small = forecast_df[forecast_df["group_id"].isin(top_groups)].copy()

# Danh sách groups dùng cho vòng for ARIMA / TFT
groups = top_groups

len(groups), groups


Top 10 group_id theo tổng 'target':
group_id
GROCERY            1642604.30
DRUG GM             423654.91
MEAT                226239.99
PRODUCE             217569.11
KIOSK-GAS           197788.38
MEAT-PCKGD          168400.29
DELI                103753.60
PASTRY               49537.94
MISC SALES TRAN      41117.99
NUTRITION            35541.83
Name: target, dtype: float64


(10,
 ['GROCERY',
  'DRUG GM',
  'MEAT',
  'PRODUCE',
  'KIOSK-GAS',
  'MEAT-PCKGD',
  'DELI',
  'PASTRY',
  'MISC SALES TRAN',
  'NUTRITION'])

In [4]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Tắt các cảnh báo rối mắt
warnings.filterwarnings("ignore")
warnings.simplefilter('ignore', ConvergenceWarning)

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    y_true=np.array(y_true); y_pred=np.array(y_pred)
    denom=np.where(y_true==0,1,y_true)
    return np.mean(np.abs((y_true-y_pred)/denom))

HORIZON=2
WINDOWS=5

arima_metrics=[]
for g in groups:
    sub = forecast_df_small[forecast_df_small.group_id==g].sort_values("time_idx")
    y = sub["target"].values
    t = sub["time_idx"].values
    max_t = t.max()
    cutoffs=[max_t-HORIZON-i*HORIZON for i in range(WINDOWS)][::-1]

    for c in cutoffs:
        train_y = y[t<=c]
        test_y  = y[(t>c)&(t<=c+HORIZON)]
        if len(train_y)<10 or len(test_y)<HORIZON: 
            continue
        try:
            fit = ARIMA(train_y, order=(1,1,1)).fit()
            pred = fit.forecast(steps=HORIZON)
        except:
            pred = np.repeat(train_y.mean(), HORIZON)

        arima_metrics.append({
            "group_id": g,
            "rmse": np.sqrt(mean_squared_error(test_y, pred)),
            "mae":  mean_absolute_error(test_y, pred),
            "mape": mape(test_y, pred)
        })

arima_metrics[:3]



[{'group_id': 'GROCERY',
  'rmse': np.float64(5438.531606992048),
  'mae': 5435.013845279569,
  'mape': np.float64(0.1256392492451597)},
 {'group_id': 'GROCERY',
  'rmse': np.float64(3832.2184538992815),
  'mae': 3008.033256530729,
  'mape': np.float64(0.06195609692111692)},
 {'group_id': 'GROCERY',
  'rmse': np.float64(6031.60797581148),
  'mae': 6031.580350660955,
  'mape': np.float64(0.1393374857310393)}]

In [5]:
from prophet import Prophet
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Đảm bảo hàm mape đã được định nghĩa từ trước
# def mape(y_true, y_pred): ...

prophet_metrics = []

for g in groups:
    # Lấy dữ liệu của từng nhóm
    sub = forecast_df_small[forecast_df_small.group_id == g].sort_values("time_idx").copy()

    # Prophet yêu cầu cột 'ds' (datetime) và 'y' (target)
    sub["ds"] = pd.to_datetime(sub["week_no"], unit="W", origin="unix")
    sub["y"]  = sub["target"]

    max_t = sub["time_idx"].max()
    # Tạo các điểm cắt (cutoffs) để test
    cutoffs = [max_t - HORIZON - i * HORIZON for i in range(WINDOWS)][::-1]

    for c in cutoffs:
        train = sub[sub.time_idx <= c][["ds", "y"]]
        test  = sub[(sub.time_idx > c) & (sub.time_idx <= c + HORIZON)]

        if len(train) < 10 or len(test) < HORIZON:
            continue

        # Huấn luyện mô hình
        mp = Prophet()
        mp.fit(train)

        # Dự báo
        future = pd.DataFrame({"ds": pd.to_datetime(test["week_no"], unit="W", origin="unix")})
        fc = mp.predict(future)
        pred = fc["yhat"].values

        # --- ĐOẠN SỬA LỖI QUAN TRỌNG ---
        # Tính MSE trước, sau đó dùng np.sqrt để ra RMSE
        rmse_val = np.sqrt(mean_squared_error(test["target"], pred))
        
        prophet_metrics.append({
            "group_id": g,
            "rmse": rmse_val,
            "mae":  mean_absolute_error(test["target"], pred),
            "mape": mape(test["target"], pred)
        })

# In kết quả
prophet_metrics[:3]


Importing plotly failed. Interactive plots will not work.


14:34:54 - cmdstanpy - INFO - Chain [1] start processing


14:34:54 - cmdstanpy - INFO - Chain [1] done processing


14:34:54 - cmdstanpy - INFO - Chain [1] start processing


14:34:54 - cmdstanpy - INFO - Chain [1] done processing


...

14:35:05 - cmdstanpy - INFO - Chain [1] start processing


14:35:06 - cmdstanpy - INFO - Chain [1] done processing


14:35:06 - cmdstanpy - INFO - Chain [1] start processing


14:35:06 - cmdstanpy - INFO - Chain [1] done processing


[{'group_id': 'GROCERY',
  'rmse': np.float64(10793.93359301018),
  'mae': 10793.922552168078,
  'mape': np.float64(0.2494084642409935)},
 {'group_id': 'GROCERY',
  'rmse': np.float64(8244.980902236568),
  'mae': 7856.179666764267,
  'mape': np.float64(0.17408894280635245)},
 {'group_id': 'GROCERY',
  'rmse': np.float64(11374.45382645973),
  'mae': 11363.533591020387,
  'mape': np.float64(0.26251767918542845)}]

In [6]:
# Make each group_id continuous in time_idx for TFT only
forecast_df_tft = forecast_df.sort_values(["group_id","time_idx"]).copy()

all_groups = forecast_df_tft["group_id"].unique()
t_min = int(forecast_df_tft["time_idx"].min())
t_max = int(forecast_df_tft["time_idx"].max())

full_index = pd.MultiIndex.from_product(
    [all_groups, np.arange(t_min, t_max+1)],
    names=["group_id","time_idx"]
)

forecast_df_tft = (
    forecast_df_tft.set_index(["group_id","time_idx"])
                   .reindex(full_index)
                   .reset_index()
)

# fill known covariates theo group
known_cols = ["avg_price","discount_rate","promo_display","promo_mailer","weekofyear","week_no"]
for c in known_cols:
    if c in forecast_df_tft.columns:
        forecast_df_tft[c] = (
            forecast_df_tft.groupby("group_id")[c]
                           .ffill()
                           .bfill()
        )

# fill target tuần thiếu (chọn 0 cho chuẩn retail)
forecast_df_tft["target"] = forecast_df_tft["target"].fillna(0)

# safety weekofyear nếu NA
forecast_df_tft["weekofyear"] = (
    forecast_df_tft["weekofyear"]
    .fillna(forecast_df_tft["time_idx"] % 52)
    .astype(int)
)

forecast_df_tft.head()


,group_id,time_idx,week_no,qty,target,retail_disc,coupon_disc,coupon_match_disc,avg_price,promo_display,promo_mailer,promo_price_red,total_disc,gross_sales,discount_rate,weekofyear
0,AUTOMOTIVE,0,1.0,1.0,4.99,0.0,0.0,0.0,4.99,0.0,0.0,0.0,0.0,4.99,0.0,1
1,AUTOMOTIVE,1,1.0,NaN,0.00,NaN,NaN,NaN,4.99,0.0,0.0,NaN,NaN,NaN,0.0,1
2,AUTOMOTIVE,2,1.0,NaN,0.00,NaN,NaN,NaN,4.99,0.0,0.0,NaN,NaN,NaN,0.0,1
3,AUTOMOTIVE,3,1.0,NaN,0.00,NaN,NaN,NaN,4.99,0.0,0.0,NaN,NaN,NaN,0.0,1
4,AUTOMOTIVE,4,1.0,NaN,0.00,NaN,NaN,NaN,4.99,0.0,0.0,NaN,NaN,NaN,0.0,1


In [7]:
import torch
from lightning.pytorch import Trainer, seed_everything   # <-- đổi dòng này
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss


seed_everything(42)

max_encoder_length = 12
max_prediction_length = 2
training_cutoff = forecast_df_tft.time_idx.max() - max_prediction_length

training = TimeSeriesDataSet(
    forecast_df_tft[forecast_df_tft.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group_id"],

    # KNOWN covariates (promo/price biết trước)
    time_varying_known_reals=[
        "time_idx","weekofyear",
        "avg_price","discount_rate",
        "promo_display","promo_mailer"
    ],

    # UNKNOWN: chỉ target
    time_varying_unknown_reals=["target"],

    target_normalizer=GroupNormalizer(groups=["group_id"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

valid = TimeSeriesDataSet.from_dataset(
    training, forecast_df_tft, predict=True, stop_randomization=True
)

train_loader = training.to_dataloader(train=True, batch_size=64, num_workers=0)
valid_loader = valid.to_dataloader(train=False, batch_size=64, num_workers=0)

tft = TemporalFusionTransformer.from_dataset(
    training,
    hidden_size=64, attention_head_size=4, dropout=0.1,
    hidden_continuous_size=64, learning_rate=1e-3,
    loss=QuantileLoss()
)

trainer = Trainer(
    max_epochs=30,
    accelerator="auto",
    devices="auto",
    gradient_clip_val=0.1
)

trainer.fit(tft, train_loader, valid_loader)


Seed set to 42


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores



   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 432    | train
3  | prescalers                         | ModuleDict                      | 1.4 K  | train
4  | static_variable_selection          | VariableSelectionNetwork        | 51.8 K | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 140 K  | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 122 K  | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 16.8 K | train
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 16.8 K 

...

Validation DataLoader 0: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 41.43it/s]

Epoch 29: 100%|███████████████████████████████████| 18/18 [00:01<00:00,  9.81it/s, v_num=19, train_loss_step=97.90, val_loss=944.0, train_loss_epoch=80.00]

Epoch 29: 100%|███████████████████████████████████| 18/18 [00:01<00:00,  9.81it/s, v_num=19, train_loss_step=97.90, val_loss=944.0, train_loss_epoch=89.10]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|███████████████████████████████████| 18/18 [00:01<00:00,  9.25it/s, v_num=19, train_loss_step=97.90, val_loss=944.0, train_loss_epoch=89.10]

In [8]:
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

prediction = tft.predict(valid_loader, mode="prediction", return_x=True)

pred = prediction.output.cpu().numpy().reshape(-1)
y_true = prediction.x["decoder_target"].cpu().numpy().reshape(-1)

rmse = np.sqrt(mean_squared_error(y_true, pred))
mae  = mean_absolute_error(y_true, pred)

# Tính MAPE (xử lý mẫu số = 0)
denom = np.where(y_true == 0, 1, y_true)
mape = np.mean(np.abs((y_true - pred) / denom))

tft_metrics = {"rmse": rmse, "mae": mae, "mape": mape}
print(tft_metrics)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


{'rmse': np.float64(4696.698840675224), 'mae': 1069.449462890625, 'mape': np.float32(2.3584862)}


In [9]:
arima_df   = pd.DataFrame(arima_metrics).assign(model="ARIMA")
prophet_df = pd.DataFrame(prophet_metrics).assign(model="Prophet")
tft_df     = pd.DataFrame([tft_metrics]).assign(model="TFT", group_id="ALL_TOP10")

metrics_all = pd.concat([arima_df, prophet_df, tft_df], ignore_index=True)
metrics_all.to_csv(r"D:\STAT3013.Q12_Group01\results\forecast_metrics.csv", index=False)

metrics_all


,group_id,rmse,mae,mape,model
0,GROCERY,5438.531607,5435.013845,0.125639,ARIMA
1,GROCERY,3832.218454,3008.033257,0.061956,ARIMA
2,GROCERY,6031.607976,6031.580351,0.139337,ARIMA
3,GROCERY,5532.119299,4659.458096,0.094531,ARIMA
4,GROCERY,33568.388314,26786.321863,3.987424,ARIMA
...,...,...,...,...,...
96,NUTRITION,75.789295,72.720798,0.068884,Prophet
97,NUTRITION,72.709599,71.735986,0.063129,Prophet
98,NUTRITION,169.324904,169.285981,0.163305,Prophet
99,NUTRITION,782.063576,647.734319,3.929288,Prophet


In [10]:
# ===== NB06: SAVE RESULTS =====
import pandas as pd
import numpy as np

# Dùng RESULTS đọc từ config (đã khai báo ở Cell 0)
OUT_DIR = RESULTS
OUT_DIR.mkdir(exist_ok=True)

# 1) ARIMA / Prophet metrics DF
arima_df   = pd.DataFrame(arima_metrics).assign(model="ARIMA")
prophet_df = pd.DataFrame(prophet_metrics).assign(model="Prophet")

# 2) TFT metrics DF (overall top10)
tft_df = pd.DataFrame([tft_metrics]).assign(model="TFT", group_id="ALL_TOP10")

# 3) Save combined forecasting metrics
metrics_all = pd.concat([arima_df, prophet_df, tft_df], ignore_index=True)
metrics_path = OUT_DIR / "forecast_metrics.csv"
metrics_all.to_csv(metrics_path, index=False)
print("Saved forecast_metrics.csv to:", metrics_path)

# 4) Save riêng TFT metrics (để report dễ trích)
tft_metrics_path = OUT_DIR / "nb06_tft_metrics.csv"
tft_df.to_csv(tft_metrics_path, index=False)
print("Saved nb06_tft_metrics.csv to:", tft_metrics_path)

# 5) Save raw TFT predictions vs actual
tft_pred_df = pd.DataFrame({"y_true": y_true, "y_pred": pred})
tft_pred_path = OUT_DIR / "nb06_tft_preds.parquet"
tft_pred_df.to_parquet(tft_pred_path, index=False)
print("Saved nb06_tft_preds.parquet to:", tft_pred_path)

metrics_all.head()


Saved forecast_metrics.csv to: D:\STAT3013.Q12_Group01\results\forecast_metrics.csv
Saved nb06_tft_metrics.csv to: D:\STAT3013.Q12_Group01\results\nb06_tft_metrics.csv
Saved nb06_tft_preds.parquet to: D:\STAT3013.Q12_Group01\results\nb06_tft_preds.parquet


,group_id,rmse,mae,mape,model
0,GROCERY,5438.531607,5435.013845,0.125639,ARIMA
1,GROCERY,3832.218454,3008.033257,0.061956,ARIMA
2,GROCERY,6031.607976,6031.580351,0.139337,ARIMA
3,GROCERY,5532.119299,4659.458096,0.094531,ARIMA
4,GROCERY,33568.388314,26786.321863,3.987424,ARIMA
